In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
SEASON_ID = 1044

url = f"https://www.ibdb.com/season/{SEASON_ID}"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print(f"Loaded season {SEASON_ID}")

Loaded season 1044


In [3]:
production_links = []

for a in soup.find_all("a", href=True):
    href = a["href"]

    match = re.search(r"/broadway-production/.*-(\d+)$", href)

    if match:
        production_links.append({
            "production_id": match.group(1),
            "title": a.get_text(strip=True),
            "url": "https://www.ibdb.com" + href
        })

df = pd.DataFrame(production_links)

print(f"Raw links found: {len(df)}")

Raw links found: 165


In [4]:
df = (
    df
    .drop_duplicates(subset="production_id")
    .sort_values("production_id")
    .reset_index(drop=True)
)

print(f"Unique productions: {len(df)}")

df.head()

Unique productions: 94


,production_id,title,url
0,1047,My Sister Eileen,https://www.ibdb.com/broadway-production/my-si...
1,1080,Claudia,https://www.ibdb.com/broadway-production/claud...
2,1113,Best Foot Forward,https://www.ibdb.com/broadway-production/best-...
3,1123,Let's Face It!,https://www.ibdb.com/broadway-production/lets-...
4,1127,Blithe Spirit,https://www.ibdb.com/broadway-production/blith...


In [5]:
assert df["production_id"].is_unique
assert df["production_id"].notna().all()

print("✓ No duplicate production IDs")
print("✓ No missing production IDs")

✓ No duplicate production IDs
✓ No missing production IDs


In [6]:
def get_season_productions(season_id):
    """
    Extract all Broadway productions listed for a season.

    Parameters
    ----------
    season_id : int or str

    Returns
    -------
    pandas.DataFrame
        Columns:
        - production_id
        - title
        - url
        - season_id
    """

    url = f"https://www.ibdb.com/season/{season_id}"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    rows = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        match = re.search(r"/broadway-production/.*-(\d+)$", href)

        if match:
            rows.append({
                "production_id": match.group(1),
                "title": a.get_text(strip=True),
                "url": "https://www.ibdb.com" + href,
                "season_id": season_id
            })

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset="production_id")
        .reset_index(drop=True)
    )

    assert df["production_id"].is_unique
    assert df["production_id"].notna().all()

    return df

In [7]:
catalogue = get_season_productions(1044)

print(f"{len(catalogue)} productions")

catalogue.head()

94 productions


,production_id,title,url,season_id
0,1203,The Cat Screams,https://www.ibdb.com/broadway-production/the-c...,1044
1,1204,"Laugh, Town, Laugh!",https://www.ibdb.com/broadway-production/laugh...,1044
2,1205,Broken Journey,https://www.ibdb.com/broadway-production/broke...,1044
3,1206,Star and Garter,https://www.ibdb.com/broadway-production/star-...,1044
4,1207,Stars on Ice,https://www.ibdb.com/broadway-production/stars...,1044
